# Temporal Example

The approach for this temporal example is the following steps:

1. Setup a base temporal table with data using commercial real estate data  
2. Split the table into two sections: SetA and SetB  
3. Add the SetA data to the base table with temporal  
4. Create the embeddings for SetA data  
5. Create the index for SetA data  
6. Show a basic search working  
7. Add data from SetB  
8. Create the embeddings for SetB and add to the base temporal table  
9. Rebuild the HNSW Index for the SetA+SetB of data  
10. Show a basic search is working + row count  
11. Show a basic search on the old data + row count  
----

In [258]:
%connect vsdemo

Connected: 'vsdemo' connection activated for user 'df120645'


In [259]:
DATABASE df120645

Success: 1 rows affected

In [260]:
DROP TABLE commercial_real_estate_temporal

Success: 27 rows affected

In [261]:
CREATE MULTISET TABLE df120645.commercial_real_estate_temporal
    ,FALLBACK
    ,NO BEFORE JOURNAL
    ,NO AFTER JOURNAL
    ,CHECKSUM = DEFAULT
    ,DEFAULT MERGEBLOCKRATIO
    ,MAP = TD_MAP2
(
    id INTEGER,
    "Unnamed: 0" VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
    "title" VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
    price VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
    nbn VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
    address VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
    text VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
    area VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
    "type" VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
    lattitude FLOAT,
    longitude FLOAT,
    Listing_Validity PERIOD(DATE) AS VALIDTIME
)
PRIMARY INDEX (id);

Success: 0 rows affected

### Show Temporal Base Table
**Step 1:** This is the base table for data

In [262]:
SHOW TABLE commercial_real_estate_temporal

,Request Text
1,"CREATE MULTISET TABLE DF120645.commercial_real_estate_temporal ,FALLBACK , NO BEFORE JOURNAL, NO AFTER JOURNAL, CHECKSUM = DEFAULT, DEFAULT MERGEBLOCKRATIO, MAP = TD_MAP2 ( id INTEGER, ""Unnamed: 0"" VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC, ""title"" VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC, price VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC, nbn VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC, address VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC, text VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC, area VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC, ""type"" VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC, lattitude FLOAT, longitude FLOAT, Listing_Validity PERIOD(DATE) AS VALIDTIME) PRIMARY INDEX ( id );"


### Note the Listing_Validity column which governs when the listing is active or valid

```sql
CREATE MULTISET TABLE df120645.commercial_real_estate_temporal
    ,FALLBACK
    ,NO BEFORE JOURNAL
    ,NO AFTER JOURNAL
    ,CHECKSUM = DEFAULT
    ,DEFAULT MERGEBLOCKRATIO
    ,MAP = TD_MAP2
(
    id INTEGER,
    "Unnamed: 0" VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
    "title" VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
    price VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
    nbn VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
    address VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
    text VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
    area VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
    "type" VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
    lattitude FLOAT,
    longitude FLOAT,
    Listing_Validity PERIOD(DATE) AS VALIDTIME
)
PRIMARY INDEX (id);
```

---
**ONLY** Execute if reloading data

In [263]:
DELETE FROM commercial_real_estate_temporal;

Success: 0 rows affected

---

**CAPTURE** the timestamp as we have to know old / new times

In [264]:
DROP TABLE commercial_real_estate_setA

Success: 25 rows affected

**Step 2:** Break the data into two tables so that we can add data as part of the exercise

In [265]:
-- Create Set A (e.g., Training Data)
CREATE TABLE commercial_real_estate_setA AS (
    SELECT * 
    FROM df120645.commercial_real_estate_example
    WHERE HASHAMP(HASHBUCKET(HASHROW(id))) MOD 10 < 7 -- Assign 70 percent of rows to Set A
) WITH DATA;

Success: 0 rows affected

**View** row count for Set A of data

In [266]:
SELECT count(*) FROM commercial_real_estate_setA;

,Count(*)
1,1389


In [267]:
DROP TABLE commercial_real_estate_setB

Success: 25 rows affected

**Create** a separate set of data SetB

In [268]:
-- Create Set B (e.g., Testing Data)
CREATE TABLE commercial_real_estate_setB AS (
    SELECT * 
    FROM df120645.commercial_real_estate_iii
    WHERE HASHAMP(HASHBUCKET(HASHROW(id))) MOD 10 >= 7 -- Assign 30 percent of rows to Set B
) WITH DATA;

Success: 0 rows affected

**View** the row count for SetB.

In [269]:
SELECT count(*) FROM df120645.commercial_real_estate_setB;

,Count(*)
1,206


**Step 3:** Add first group (SetA) of rows to the base temporal table using current time

In [270]:
NONSEQUENCED VALIDTIME
INSERT INTO df120645.commercial_real_estate_temporal 
    (id, "Unnamed: 0", "title", price, nbn, address, "text", area, "type", lattitude, longitude, Listing_Validity)
SELECT 
    id, "Unnamed: 0", "title", price, nbn, address, "text", area, "type", lattitude, longitude,
    PERIOD(DATE '2025-05-15', UNTIL_CHANGED)
FROM 
    df120645.commercial_real_estate_setA;

Success: 1389 rows affected

**Validate** the row count added to the empty table.

In [271]:
 set session validtime ( PERIOD(TIMESTAMP '2025-01-01 00:00:00.000', TIMESTAMP '2025-05-16 00:00:00.000'));

Success: 1 rows affected

In [272]:
SELECT count(*) FROM commercial_real_estate_temporal;

,Count(*),VALIDTIME
1,1389,"2025-05-15 00:00:00.000000,2025-05-16 00:00:00.000000"


In [ ]:
select * from df120645.commercial_real_estate_temporal sample 10;

,id,Unnamed: 0,title,price,nbn,address,text,area,type,lattitude,longitude,VALIDTIME
1,469,469,"707/37 Bligh Street, Sydney, NSW 2000 - For Sale - Offices - ID: 380769",Under Offer,Service Available - Fibre to the building (FTTB),"37 Bligh Street, Sydney, NSW","Noonan Property has been exclusively appointed to offer Suite 707, 37 Bligh Street, Sydney for sale. - Premium office suite of the highest quality - Set in one of Sydney's best strata buildings - Truly unique and irreplaceable opportunity - Professional and executive-style workspace - Prime corner positioning and large windows - Huge frontages to Bligh and Hunter Streets - Attractive views and excellent natural light - Views overlooking Richard Johnson Square - Modern fitout with a separate boardroom - Fibre 400 up and down internet available - On floor bathroom and kitchen amenities - Highly admired commercial strata building - Striking main lobby with high spec finishes - Excellent building services including 3 lifts - Blue ribbon location in core CBD precinct - Surrounded by Sydney's finest restaurants - Minutes to Martin Place and Circular Quay - Easy walking distance to Wynyard Station - Opposite the Martin Place Metro Station - Popular, exclusive and tightly held location - Collective sale and developme",61m²,,-33.8659187,151.2097205,"2025-05-15 00:00:00.000000,2025-05-16 00:00:00.000000"
2,265,265,"104 Walker Street, Dandenong, VIC 3175 - For Sale - Retail - ID: 220791",$435K,Build Commenced,"104 Walker Street, Dandenong, VIC","Ideally located in the CBD of Dandenong is this highly exposed property which is now available for Sale as a leased investment. Lease Details: Rental: $24,000.00pa plus GST and Outgoings Start: 20th July 2015 Term: Three (3) Years Further Term: Three (3) Years Bond: $4,400.00 Reviews: 3% annually Use: Photography and Digital Print Business Building area: 111m2* Features: -Excellent retail / office potential -High traffic exposure -short stroll to the Dandenong Plaza Contact Blake Quilligan on 0432 929 073 to inspect today. *approx.",111m²,,-37.9881736,145.2148967,"2025-05-15 00:00:00.000000,2025-05-16 00:00:00.000000"
3,999,998,"Units 1-6/20 Service Street, Maroochydore, QLD 4558 - For Sale - Industrial - ID: 369367","$455,715 + GST (if applicable)",Not Currently Available,"20 Service Street, Maroochydore, QLD",+ Unit 6 - last available + Warehouse area 160sqm + 3 Phase power + Roller door access + Close to Maroochydore CBD Call or email David C Smith to inspect the property or find out about other suitable options.,160m²,,-26.6501645,153.0614488,"2025-05-15 00:00:00.000000,2025-05-16 00:00:00.000000"
4,938,937,"9, 2 Money Close, Rouse Hill, NSW 2155 - For Sale - Industrial - ID: 383801","$752,500 + GST",Not Currently Available,"2 Money Close, Rouse Hill, NSW",Money Business Park is a premium development consisting of 22 units plus an onsite cafe. Key features: • Potential investment opportunity reflecting 4% net return • Excellent truck access • Information Memorandum available upon request Bawdens ID: 55046,172m²,,-33.6711342,150.9254538,"2025-05-15 00:00:00.000000,2025-05-16 00:00:00.000000"
5,1407,1406,"2/4 Chisholm Court, Wodonga, VIC 3690 - Sold - Investment","$140,000",Service Available,"4 Chisholm Court, Wodonga, VIC","- Neat colourbond shed with roller door - Internal amenities - 3-Phase power - Small side security yard - Court location, handy to Wodonga CBD - Net rental: $9,225.60 pa plus GST",120m²,Industrial,-36.1349743,146.9028473,"2025-05-15 00:00:00.000000,2025-05-16 00:00:00.000000"
6,326,326,"Shop 1, 78 Keilor Road, Essendon North, VIC 3041 - Sale / Lease - Retail - ID: 379103",Contact Agent,Service Available - Fibre to the building (FTTB),"78 Keilor Road, Essendon North, VIC","POINT OF INTEREST: Light and bright, ground floor retail investment. - Located in the boutique Erantis Development - Highly exposed with the Erantis Building comprising 30m* of street frontage to Keilor Road - Currently leased to Prolash Pry Ltd retu

### Create Embeddings for SetA data

<img src="VSData1b.png" alt="Alt Text" width="50%">

**Step 4:** Here we create a temporary holding table for the embeddings to insert into the final table

---
**Only** for re-execution

In [274]:
DROP TABLE vectorstore_commercial_real_estate_temporal_index_temp

Success: 19 rows affected

---
**Create** Embeddings into staging table

In [275]:
CREATE MULTISET TABLE vectorstore_commercial_real_estate_temporal_index_temp AS (
SELECT id, price, Embedding as Vector_Index, Message 
FROM AI_TEXTEMBEDDINGS(   
    ON (
        SELECT 
            dt.id, 
            CONCAT(
                dt.text, ' ', 
                dt.address, ' ', 
                'Price: ', CAST(dt.price AS VARCHAR(50))
            ) AS text, 
            dt.price, 
            dt."title", 
            td_byone() 
        FROM commercial_real_estate_iii dt 
        -- SAMPLE 1
    ) AS InputTable PARTITION BY TD_BYONE()
    USING 
        authorization(AWSEmbeddingsAuth)
        TextColumn('text')
        ApiType('aws')
        REGION('us-west-2')
        ModelName('amazon.titan-embed-text-v1')
        outputformat('vector')
        Accumulate(' "id" ', ' "title" ', ' "price" ')
) AS dt
) WITH DATA PRIMARY INDEX (id);

Success: 0 rows affected

**View** resulting staging table

In [276]:
SHOW TABLE vectorstore_commercial_real_estate_temporal_index_temp

,Request Text
1,"CREATE MULTISET TABLE DF120645.vectorstore_commercial_real_estate_temporal_index_temp ,FALLBACK , NO BEFORE JOURNAL, NO AFTER JOURNAL, CHECKSUM = DEFAULT, DEFAULT MERGEBLOCKRATIO, MAP = TD_MAP2 ( id INTEGER, price VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC, Vector_Index SYSUDTLIB.Vector, Message VARCHAR(32000) CHARACTER SET UNICODE NOT CASESPECIFIC) PRIMARY INDEX ( id );"


```sql
CREATE MULTISET TABLE DF120645.vectorstore_commercial_real_estate_temporal_index_temp ,FALLBACK ,
     NO BEFORE JOURNAL,
     NO AFTER JOURNAL,
     CHECKSUM = DEFAULT,
     DEFAULT MERGEBLOCKRATIO,
     MAP = TD_MAP2
     (
      id INTEGER,
      price VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
      Vector_Index SYSUDTLIB.Vector,
      Message VARCHAR(32000) CHARACTER SET UNICODE NOT CASESPECIFIC)
PRIMARY INDEX ( id );
```

**View** a sample of the data

In [277]:
SELECT id, price, vector_index, vector_index as vector_index_normalized, Message 
            FROM vectorstore_commercial_real_estate_temporal_index_temp sample 1
            WHERE Vector_Index IS NOT NULL

,id,price,Vector_Index,vector_index_normalized,Message
1,167,"$210,000",...,...,


---
**Only** drop if re-executing

In [278]:
DROP TABLE vectorstore_commercial_real_estate_temporal_index_temp_II

Success: 20 rows affected

---
Create the normalized vector column

In [279]:
CREATE MULTISET TABLE vectorstore_commercial_real_estate_temporal_index_temp_II AS (
SELECT id, price, vector_index, vector_index_normalized, Message 
FROM TD_Vectornormalize(
    ON (
           SELECT id, price, vector_index, vector_index as vector_index_normalized, Message 
            FROM vectorstore_commercial_real_estate_temporal_index_temp 
            WHERE vector_index IS NOT NULL
    ) AS InputTable
                USING
                IDColumns('id')
                TargetColumns('vector_index_normalized')
                Accumulate('price', 'vector_index', 'message')
                Approach('UNITVECTOR')
                EmbeddingSize(1536)
                ) AS dt
) WITH DATA PRIMARY INDEX (id);

Success: 0 rows affected

**View** table with normalized vector colum

In [280]:
SHOW TABLE vectorstore_commercial_real_estate_temporal_index_temp_II

,Request Text
1,"CREATE MULTISET TABLE DF120645.vectorstore_commercial_real_estate_temporal_index_temp_II ,FALLBACK , NO BEFORE JOURNAL, NO AFTER JOURNAL, CHECKSUM = DEFAULT, DEFAULT MERGEBLOCKRATIO, MAP = TD_MAP2 ( id INTEGER, price VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC, vector_index SYSUDTLIB.Vector, vector_index_normalized SYSUDTLIB.Vector, Message VARCHAR(32000) CHARACTER SET UNICODE NOT CASESPECIFIC) PRIMARY INDEX ( id );"


```sql
CREATE MULTISET TABLE DF120645.vectorstore_commercial_real_estate_temporal_index_temp_II ,FALLBACK ,
     NO BEFORE JOURNAL,
     NO AFTER JOURNAL,
     CHECKSUM = DEFAULT,
     DEFAULT MERGEBLOCKRATIO,
     MAP = TD_MAP2
     (
      id INTEGER,
      price VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
      vector_index SYSUDTLIB.Vector,
      vector_index_normalized SYSUDTLIB.Vector,
      Message VARCHAR(32000) CHARACTER SET UNICODE NOT CASESPECIFIC)
PRIMARY INDEX ( id );
```

### Show Temporal Index Table
Show what the table holding the data looks like

In [281]:
SHOW TABLE vectorstore_commercial_real_estate_temporal_index_temp

,Request Text
1,"CREATE MULTISET TABLE DF120645.vectorstore_commercial_real_estate_temporal_index_temp ,FALLBACK , NO BEFORE JOURNAL, NO AFTER JOURNAL, CHECKSUM = DEFAULT, DEFAULT MERGEBLOCKRATIO, MAP = TD_MAP2 ( id INTEGER, price VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC, Vector_Index SYSUDTLIB.Vector, Message VARCHAR(32000) CHARACTER SET UNICODE NOT CASESPECIFIC) PRIMARY INDEX ( id );"


```sql
CREATE MULTISET TABLE DF120645.vectorstore_commercial_real_estate_temporal_index_temp ,FALLBACK ,
     NO BEFORE JOURNAL,
     NO AFTER JOURNAL,
     CHECKSUM = DEFAULT,
     DEFAULT MERGEBLOCKRATIO,
     MAP = TD_MAP2
     (
      id INTEGER,
      price VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
      Vector_Index SYSUDTLIB.Vector,
      Message VARCHAR(32000) CHARACTER SET UNICODE NOT CASESPECIFIC)
PRIMARY INDEX ( id );
```

In [282]:
DROP TABLE vectorstore_commercial_real_estate_temporal_index

Success: 22 rows affected

**Add** the embeddings to the final temporal index table
As a create:

In [283]:
CREATE MULTISET TABLE vectorstore_commercial_real_estate_temporal_index (
      id,
      price,
      vector_index,
      vector_index_normalized,
      Message ,
      Listing_Validity 
      ) AS (
      NONSEQUENCED VALIDTIME PERIOD (DATE '2025-05-15', UNTIL_CHANGED)
      SELECT *
      FROM vectorstore_commercial_real_estate_temporal_index_temp_II)
   WITH DATA PRIMARY INDEX(id);

Success: 0 rows affected

**Show** the final temporal index table

In [284]:
SHOW TABLE vectorstore_commercial_real_estate_temporal_index


,Request Text
1,"CREATE MULTISET TABLE DF120645.vectorstore_commercial_real_estate_temporal_index ,FALLBACK , NO BEFORE JOURNAL, NO AFTER JOURNAL, CHECKSUM = DEFAULT, DEFAULT MERGEBLOCKRATIO, MAP = TD_MAP2 ( id INTEGER, price VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC, vector_index SYSUDTLIB.Vector, vector_index_normalized SYSUDTLIB.Vector, Message VARCHAR(32000) CHARACTER SET UNICODE NOT CASESPECIFIC, Listing_Validity PERIOD(DATE) AS VALIDTIME) PRIMARY INDEX ( id );"


```sql
CREATE MULTISET TABLE DF120645.vectorstore_commercial_real_estate_temporal_index ,FALLBACK ,
     NO BEFORE JOURNAL,
     NO AFTER JOURNAL,
     CHECKSUM = DEFAULT,
     DEFAULT MERGEBLOCKRATIO,
     MAP = TD_MAP2
     (
      id INTEGER,
      price VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
      vector_index SYSUDTLIB.Vector,
      vector_index_normalized SYSUDTLIB.Vector,
      Message VARCHAR(32000) CHARACTER SET UNICODE NOT CASESPECIFIC,
      Listing_Validity PERIOD(DATE) AS VALIDTIME)
PRIMARY INDEX ( id );
```

---
**Only*** Drop if re-executing

In [285]:
DROP TABLE vectorstore_commercial_real_estate_kmeans_model_temp

Success: 20 rows affected

---
**Congratulations!** The data has been stored in a temporal table and the embeddings index has been placed in a temporal index.


# Setup VS Kmeans Model

<img src="VSDataModel2.png" alt="Alt Text" width="50%">


**Step 5:** Create the Model on the temporal table with the SetA data  

In [286]:
CREATE MULTISET TABLE vectorstore_commercial_real_estate_kmeans_model_temp as (
SELECT * FROM TD_KMEANS (
  ON vectorstore_commercial_real_estate_temporal_index AS InputTable
  USING
  IdColumn('id')
  TargetColumns('Vector_Index')
  InitialCentroidsMethod('RANDOM')
  NumClusters(10)
  Seed(0)
  StopThreshold(0.0395)
  MaxIterNum(10)
  NumInit(1)
  EmbeddingSize(1536)
) AS dt) WITH DATA NO PRIMARY INDEX

Success: 0 rows affected

In [287]:
SHOW TABLE vectorstore_commercial_real_estate_kmeans_model_temp

,Request Text
1,"CREATE MULTISET TABLE DF120645.vectorstore_commercial_real_estate_kmeans_model_temp ,FALLBACK , NO BEFORE JOURNAL, NO AFTER JOURNAL, CHECKSUM = DEFAULT, DEFAULT MERGEBLOCKRATIO, MAP = TD_MAP2 ( td_clusterid_kmeans BIGINT, vector_index SYSUDTLIB.Vector, td_size_kmeans BIGINT, td_withinss_kmeans FLOAT, id BYTEINT, td_modelinfo_kmeans VARCHAR(128) CHARACTER SET LATIN NOT CASESPECIFIC) NO PRIMARY INDEX ;"


```sql
CREATE MULTISET TABLE DF120645.vectorstore_commercial_real_estate_kmeans_model_temp ,FALLBACK ,
     NO BEFORE JOURNAL,
     NO AFTER JOURNAL,
     CHECKSUM = DEFAULT,
     DEFAULT MERGEBLOCKRATIO,
     MAP = TD_MAP2
     (
      td_clusterid_kmeans BIGINT,
      vector_index SYSUDTLIB.Vector,
      td_size_kmeans BIGINT,
      td_withinss_kmeans FLOAT,
      id BYTEINT,
      td_modelinfo_kmeans VARCHAR(128) CHARACTER SET LATIN NOT CASESPECIFIC)
NO PRIMARY INDEX ;
```

---
**Only** drop if re-executing

In [288]:
DROP TABLE vectorstore_commercial_real_estate_temporal_model

Success: 23 rows affected

**Create** the final model table with temporal 

In [289]:
   CREATE MULTISET TABLE df120645.vectorstore_commercial_real_estate_temporal_model (
      td_clusterid_kmeans,
      vector_index,
      td_size_kmeans,
      td_withinss_kmeans,
      id,
      td_modelinfo_kmeans,
      Listing_Validity 
      ) AS (
      NONSEQUENCED VALIDTIME PERIOD(DATE '2025-05-15', UNTIL_CHANGED)
      SELECT *
      FROM vectorstore_commercial_real_estate_kmeans_model_temp)
   WITH DATA
   PRIMARY INDEX(td_clusterid_kmeans);

Success: 0 rows affected

### Show Temporal Model Table

In [290]:
SHOW TABLE vectorstore_commercial_real_estate_temporal_model

,Request Text
1,"CREATE MULTISET TABLE DF120645.vectorstore_commercial_real_estate_temporal_model ,FALLBACK , NO BEFORE JOURNAL, NO AFTER JOURNAL, CHECKSUM = DEFAULT, DEFAULT MERGEBLOCKRATIO, MAP = TD_MAP2 ( td_clusterid_kmeans BIGINT, vector_index SYSUDTLIB.Vector, td_size_kmeans BIGINT, td_withinss_kmeans FLOAT, id BYTEINT, td_modelinfo_kmeans VARCHAR(128) CHARACTER SET LATIN NOT CASESPECIFIC, Listing_Validity PERIOD(DATE) AS VALIDTIME) PRIMARY INDEX ( td_clusterid_kmeans );"


```sql
CREATE MULTISET TABLE DF120645.vectorstore_commercial_real_estate_temporal_model ,FALLBACK ,
     NO BEFORE JOURNAL,
     NO AFTER JOURNAL,
     CHECKSUM = DEFAULT,
     DEFAULT MERGEBLOCKRATIO,
     MAP = TD_MAP2
     (
      td_clusterid_kmeans BIGINT,
      vector_index SYSUDTLIB.Vector,
      td_size_kmeans BIGINT,
      td_withinss_kmeans FLOAT,
      id BYTEINT,
      td_modelinfo_kmeans VARCHAR(128) CHARACTER SET LATIN NOT CASESPECIFIC,
      Listing_Validity PERIOD(DATE) AS VALIDTIME)
PRIMARY INDEX ( td_clusterid_kmeans );
```

In [291]:
DROP TABLE vectorstore_commercial_real_estate_temporal_centroids_temp

Success: 18 rows affected

---
**Create** centroids staging table

In [292]:
CREATE MULTISET TABLE vectorstore_commercial_real_estate_temporal_centroids_temp AS (
SELECT id, Vector_Index, td_clusterid_kmeans as clusterID FROM TD_KMEANSPREDICT(
  ON vectorstore_commercial_real_estate_temporal_index AS InputTable
  ON vectorstore_commercial_real_estate_kmeans_model_temp AS ModelTable DIMENSION
  USING
  Accumulate( 'Vector_Index')
) AS dt) WITH DATA PRIMARY INDEX (clusterID);

Success: 0 rows affected

In [293]:
SHOW TABLE vectorstore_commercial_real_estate_temporal_centroids_temp

,Request Text
1,"CREATE MULTISET TABLE DF120645.vectorstore_commercial_real_estate_temporal_centroids_temp ,FALLBACK , NO BEFORE JOURNAL, NO AFTER JOURNAL, CHECKSUM = DEFAULT, DEFAULT MERGEBLOCKRATIO, MAP = TD_MAP2 ( id INTEGER, vector_index SYSUDTLIB.Vector, clusterID BIGINT) PRIMARY INDEX ( clusterID );"


```sql
CREATE MULTISET TABLE DF120645.vectorstore_commercial_real_estate_temporal_centroids_temp ,FALLBACK ,
     NO BEFORE JOURNAL,
     NO AFTER JOURNAL,
     CHECKSUM = DEFAULT,
     DEFAULT MERGEBLOCKRATIO,
     MAP = TD_MAP2
     (
      id INTEGER,
      vector_index SYSUDTLIB.Vector,
      clusterID BIGINT)
PRIMARY INDEX ( clusterID );
```

In [294]:
DROP TABLE vectorstore_commercial_real_estate_temporal_centroids

Success: 20 rows affected

---
**Move** data into temporal centroids table

In [295]:
CREATE MULTISET TABLE df120645.vectorstore_commercial_real_estate_temporal_centroids (
      id,
      vector_index,
      clusterID,
      Listing_Validity 
      ) AS (
      NONSEQUENCED VALIDTIME PERIOD(DATE '2025-05-15', UNTIL_CHANGED)
      SELECT *
      FROM vectorstore_commercial_real_estate_temporal_centroids_temp)
   WITH DATA
   PRIMARY INDEX(id);

Success: 0 rows affected

### Show Temporal Centroids Table

In [296]:
SHOW TABLE vectorstore_commercial_real_estate_temporal_centroids

,Request Text
1,"CREATE MULTISET TABLE DF120645.vectorstore_commercial_real_estate_temporal_centroids ,FALLBACK , NO BEFORE JOURNAL, NO AFTER JOURNAL, CHECKSUM = DEFAULT, DEFAULT MERGEBLOCKRATIO, MAP = TD_MAP2 ( id INTEGER, vector_index SYSUDTLIB.Vector, clusterID BIGINT, Listing_Validity PERIOD(DATE) AS VALIDTIME) PRIMARY INDEX ( id );"


```sql
CREATE MULTISET TABLE DF120645.vectorstore_commercial_real_estate_temporal_centroids ,FALLBACK ,
     NO BEFORE JOURNAL,
     NO AFTER JOURNAL,
     CHECKSUM = DEFAULT,
     DEFAULT MERGEBLOCKRATIO,
     MAP = TD_MAP2
     (
      id INTEGER,
      vector_index SYSUDTLIB.Vector,
      clusterID BIGINT)
PRIMARY INDEX ( clusterID );
```

In [297]:
DROP TABLE question_vector

Success: 17 rows affected

### Search

Form a question

In [298]:
CREATE MULTISET TABLE question_vector  AS (
SELECT id, Embedding 
FROM AI_TEXTEMBEDDINGS(   
    ON (
        SELECT
            1 as ID,
            'I need properties on Harrington St.' as "Text"
    ) AS InputTable 
    USING 
        authorization(AWSEmbeddingsAuth)
        TextColumn('text')
        ApiType('aws')
        REGION('us-west-2')
        ModelName('amazon.titan-embed-text-v1')
        outputformat('vector')
        Accumulate(' "id" ')
) AS dt
) WITH DATA PRIMARY INDEX (id);

Success: 0 rows affected

In [299]:
SHOW TABLE question_vector

,Request Text
1,"CREATE MULTISET TABLE DF120645.question_vector ,FALLBACK , NO BEFORE JOURNAL, NO AFTER JOURNAL, CHECKSUM = DEFAULT, DEFAULT MERGEBLOCKRATIO, MAP = TD_MAP2 ( ID BYTEINT, Embedding SYSUDTLIB.Vector) PRIMARY INDEX ( ID );"


```sql
CREATE MULTISET TABLE DF120645.question_vector ,FALLBACK ,
     NO BEFORE JOURNAL,
     NO AFTER JOURNAL,
     CHECKSUM = DEFAULT,
     DEFAULT MERGEBLOCKRATIO,
     MAP = TD_MAP2
     (
      ID BYTEINT,
      Embedding SYSUDTLIB.Vector)
PRIMARY INDEX ( ID );
```

---
**Ask** question of data

In [300]:
WITH ClusterIDs AS (
    -- Fetch clusterID values dynamically
    SELECT reference_id FROM TD_VECTORDISTANCE(
                ON question_vector AS TargetTable DIMENSION
                ON (
                    CURRENT VALIDTIME
                    SELECT td_clusterid_kmeans as clusterID, vector_index AS centroid 
                        FROM vectorstore_commercial_real_estate_temporal_model 
                        WHERE td_clusterid_kmeans IS NOT NULL) AS ReferenceTable
                USING
                TargetIDColumn('ID')
                TargetFeatureColumns('Embedding')
                RefIDColumn('clusterID')
                RefFeatureColumns('centroid')
                DistanceMeasure('EUCLIDEAN')
                OutputSimilarity('t')
                Topk(3)
                LargeReferenceInput('t')
                EmbeddingSize(1536)
                ) AS dt 
)
CURRENT VALIDTIME
SELECT t.id, t."title", t.price, vd.similarity
FROM df120645.commercial_real_estate_temporal t
JOIN (
    SELECT reference_id, similarity
    FROM TD_VECTORDISTANCE(
        ON question_vector AS TargetTable DIMENSION
        ON (
            CURRENT VALIDTIME
            SELECT * 
            FROM vectorstore_commercial_real_estate_temporal_centroids 
            WHERE clusterID IN (SELECT reference_id FROM ClusterIDs)
        ) AS ReferenceTable
        USING
            TargetIDColumn('id')
            TargetFeatureColumns('embedding')
            RefIDColumn('id')
            RefFeatureColumns('vector_index')
            DistanceMeasure('euclidean')
            OutputSimilarity('t')
            Topk(10)
            EmbeddingSize(1536)
            LargeReferenceInput('t')
    ) AS dt1
) vd
ON t.id = vd.reference_id
ORDER BY vd.similarity DESC;

,id,title,price,similarity
1,68,"Suite 220-221, 111 Harrington Street, Sydney, nsw 2000 - For Sale - Offices - ID: 381123","$1,500,000",0.06693688177180236
2,757,"57C Carrington Street, Palmyra, WA 6157 - For Sale - Retail",Contact Agent,0.06066465198962686
3,899,"7 Durham Street, Mount Druitt, NSW 2770 - For Sale - Other",Contact the exclusive agents,0.06058482696977657
4,382,"Dunolly, VIC 3472 - For Sale - Retail","$309,000",0.06029512969880231
5,906,"265 & 267-269 Hutt Street, Adelaide, SA 5000 - For Sale - Offices - ID: 383448",Contact Agent,0.06021104332647583
6,525,"6, 55 Simcock Street, Somerville, VIC 3912 - For Sale - Industrial - ID: 386739","$480,000 (plus GST if applicable)",0.060013886526017135
7,1342,"Unit 11, 400 Canterbury Road, Surrey Hills, VIC 3127 - For Sale - Other - ID: 330210","$1,188,000",0.059966099188303854
8,1101,"11-19 Hilton Terrace, Tewantin, QLD 4565 - Sale / Lease - Retail - ID: 373500",Contact Agent,0.059774976774435926
9,882,"112 & 113, 12 Salonika Street, Parap, NT 0820 - For Sale - Retail - ID: 345843","$639,000 + GST",0.059706134452113584


# Update data, index, and Model

### Add SetB data to Base Table

In [301]:
NONSEQUENCED VALIDTIME
INSERT INTO df120645.commercial_real_estate_temporal 
    (id, "Unnamed: 0", "title", price, nbn, address, "text", area, "type", lattitude, longitude, Listing_Validity)
SELECT 
    id, "Unnamed: 0", "title", price, nbn, address, "text", area, "type", lattitude, longitude,
    PERIOD(DATE '2025-06-01', DATE '9999-01-01')
FROM 
    df120645.commercial_real_estate_setB;

Success: 206 rows affected

### Show row count with all rows

In [302]:
CURRENT VALIDTIME
SELECT Count(*) from commercial_real_estate_temporal;

,Count(*)
1,1595


### Show row count in the past

In [303]:
SEQUENCED VALIDTIME PERIOD(timestamp'2025-01-01 00:00:00.000000+00:00', timestamp'2025-05-30 00:00:00.000000+00:00')
SELECT Count(*) from commercial_real_estate_temporal;

,Count(*),VALIDTIME
1,1389,"2025-05-15 00:00:00.000000+00:00,2025-05-30 00:00:00.000000+00:00"


### Update Index

---
**Only** drop if re-executing

In [304]:
DROP TABLE vectorstore_commercial_real_estate_temporal_index_temp

Success: 19 rows affected

---
**Add** Data from SetB with embeddings into temporal_index_temp

In [305]:
CREATE MULTISET TABLE vectorstore_commercial_real_estate_temporal_index_temp AS (
SELECT id, price, Embedding as Vector_Index, Message 
FROM AI_TEXTEMBEDDINGS(   
    ON (
        SELECT 
            dt.id, 
            CONCAT(
                dt.text, ' ', 
                dt.address, ' ', 
                'Price: ', CAST(dt.price AS VARCHAR(50))
            ) AS text, 
            dt.price, 
            dt."title", 
            td_byone() 
        FROM commercial_real_estate_setB dt 
        -- SAMPLE 1
    ) AS InputTable PARTITION BY TD_BYONE()
    USING 
        authorization(AWSEmbeddingsAuth)
        TextColumn('text')
        ApiType('aws')
        REGION('us-west-2')
        ModelName('amazon.titan-embed-text-v1')
        outputformat('vector')
        Accumulate(' "id" ', ' "title" ', ' "price" ')
) AS dt
) WITH DATA PRIMARY INDEX (id);

Success: 0 rows affected

In [306]:
DROP TABLE vectorstore_commercial_real_estate_temporal_index_temp_II

Success: 20 rows affected

**Add** the normalized vectors

In [307]:
CREATE MULTISET TABLE vectorstore_commercial_real_estate_temporal_index_temp_II AS (
SELECT id, price, vector_index, vector_index_normalized, Message 
FROM TD_Vectornormalize(
    ON (
           SELECT id, price, vector_index, vector_index as vector_index_normalized, Message 
            FROM vectorstore_commercial_real_estate_temporal_index_temp 
            WHERE vector_index IS NOT NULL
    ) AS InputTable
                USING
                IDColumns('id')
                TargetColumns('vector_index_normalized')
                Accumulate('price', 'vector_index', 'message')
                Approach('UNITVECTOR')
                EmbeddingSize(1536)
                ) AS dt
) WITH DATA PRIMARY INDEX (id);

Success: 0 rows affected

In [308]:
show table vectorstore_commercial_real_estate_temporal_index;

,Request Text
1,"CREATE MULTISET TABLE DF120645.vectorstore_commercial_real_estate_temporal_index ,FALLBACK , NO BEFORE JOURNAL, NO AFTER JOURNAL, CHECKSUM = DEFAULT, DEFAULT MERGEBLOCKRATIO, MAP = TD_MAP2 ( id INTEGER, price VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC, vector_index SYSUDTLIB.Vector, vector_index_normalized SYSUDTLIB.Vector, Message VARCHAR(32000) CHARACTER SET UNICODE NOT CASESPECIFIC, Listing_Validity PERIOD(DATE) AS VALIDTIME) PRIMARY INDEX ( id );"


```sql
CREATE MULTISET TABLE DF120645.vectorstore_commercial_real_estate_temporal_index ,FALLBACK ,
     NO BEFORE JOURNAL,
     NO AFTER JOURNAL,
     CHECKSUM = DEFAULT,
     DEFAULT MERGEBLOCKRATIO,
     MAP = TD_MAP2
     (
      id INTEGER,
      price VARCHAR(1024) CHARACTER SET UNICODE NOT CASESPECIFIC,
      vector_index SYSUDTLIB.Vector,
      vector_index_normalized SYSUDTLIB.Vector,
      Message VARCHAR(32000) CHARACTER SET UNICODE NOT CASESPECIFIC,
      Listing_Validity PERIOD(DATE) AS VALIDTIME)
PRIMARY INDEX ( id );
```

**Move** the embeddings into the temporal table correctly

In [309]:
NONSEQUENCED VALIDTIME
INSERT INTO vectorstore_commercial_real_estate_temporal_index 
    (id, price, vector_index, vector_index_normalized, Message, Listing_Validity)
SELECT 
    id, price, vector_index, vector_index_normalized, Message,
    PERIOD(DATE '2025-06-01', DATE '9999-01-01')
FROM 
    vectorstore_commercial_real_estate_temporal_index_temp_II;

Success: 203 rows affected

**View** row count

In [310]:
CURRENT VALIDTIME
SELECT Count(*) from df120645.vectorstore_commercial_real_estate_temporal_index;

,Count(*)
1,1744


**Show** past row count

In [311]:
SEQUENCED VALIDTIME PERIOD(date'2011-01-01', date'2025-05-16')
SELECT count(*) from vectorstore_commercial_real_estate_temporal_index;

,Count(*),VALIDTIME
1,1541,"2025-05-15,2025-05-16"


**Setup** model steps

In [312]:
DROP TABLE vectorstore_commercial_real_estate_kmeans_model_temp

Success: 20 rows affected

**Build** model table

In [313]:
CREATE MULTISET TABLE vectorstore_commercial_real_estate_kmeans_model_temp as (
SELECT * FROM TD_KMEANS (
  ON vectorstore_commercial_real_estate_temporal_index AS InputTable
  USING
  IdColumn('id')
  TargetColumns('Vector_Index')
  InitialCentroidsMethod('RANDOM')
  NumClusters(10)
  Seed(0)
  StopThreshold(0.0395)
  MaxIterNum(10)
  NumInit(1)
  EmbeddingSize(1536)
) AS dt) WITH DATA NO PRIMARY INDEX

Success: 0 rows affected

**Delete** model data

In [314]:
DELETE df120645.vectorstore_commercial_real_estate_temporal_model

Success: 16 rows affected

In [315]:
SHOW TABLE vectorstore_commercial_real_estate_temporal_model

,Request Text
1,"CREATE MULTISET TABLE DF120645.vectorstore_commercial_real_estate_temporal_model ,FALLBACK , NO BEFORE JOURNAL, NO AFTER JOURNAL, CHECKSUM = DEFAULT, DEFAULT MERGEBLOCKRATIO, MAP = TD_MAP2 ( td_clusterid_kmeans BIGINT, vector_index SYSUDTLIB.Vector, td_size_kmeans BIGINT, td_withinss_kmeans FLOAT, id BYTEINT, td_modelinfo_kmeans VARCHAR(128) CHARACTER SET LATIN NOT CASESPECIFIC, Listing_Validity PERIOD(DATE) AS VALIDTIME) PRIMARY INDEX ( td_clusterid_kmeans );"


```sql
CREATE MULTISET TABLE DF120645.vectorstore_commercial_real_estate_temporal_model ,FALLBACK ,
     NO BEFORE JOURNAL,
     NO AFTER JOURNAL,
     CHECKSUM = DEFAULT,
     DEFAULT MERGEBLOCKRATIO,
     MAP = TD_MAP2
     (
      td_clusterid_kmeans BIGINT,
      vector_index SYSUDTLIB.Vector,
      td_size_kmeans BIGINT,
      td_withinss_kmeans FLOAT,
      id BYTEINT,
      td_modelinfo_kmeans VARCHAR(128) CHARACTER SET LATIN NOT CASESPECIFIC,
      Listing_Validity PERIOD(DATE) AS VALIDTIME)
PRIMARY INDEX ( td_clusterid_kmeans );
```

**Move** model data into temporal model table

In [317]:
NONSEQUENCED VALIDTIME
INSERT INTO vectorstore_commercial_real_estate_temporal_model 
    (td_clusterid_kmeans, vector_index, td_size_kmeans, td_withinss_kmeans, id, td_modelinfo_kmeans, Listing_Validity)
SELECT 
    td_clusterid_kmeans, vector_index, td_size_kmeans, td_withinss_kmeans, id, td_modelinfo_kmeans,
    PERIOD(DATE '2025-06-01', DATE '9999-01-01')
FROM 
    vectorstore_commercial_real_estate_kmeans_model_temp;

Success: 16 rows affected

In [318]:
DROP TABLE vectorstore_commercial_real_estate_temporal_centroids_temp

Success: 18 rows affected

**Create** temporary table for centroids

In [319]:
CREATE MULTISET TABLE vectorstore_commercial_real_estate_temporal_centroids_temp AS (
SELECT id, Vector_Index, td_clusterid_kmeans as clusterID FROM TD_KMEANSPREDICT(
  ON vectorstore_commercial_real_estate_temporal_index AS InputTable
  ON vectorstore_commercial_real_estate_kmeans_model_temp AS ModelTable DIMENSION
  USING
  Accumulate( 'Vector_Index')
) AS dt) WITH DATA PRIMARY INDEX (clusterID);

Success: 0 rows affected

**Move** centroids into temporal table

In [320]:
SHOW TABLE vectorstore_commercial_real_estate_temporal_centroids

,Request Text
1,"CREATE MULTISET TABLE DF120645.vectorstore_commercial_real_estate_temporal_centroids ,FALLBACK , NO BEFORE JOURNAL, NO AFTER JOURNAL, CHECKSUM = DEFAULT, DEFAULT MERGEBLOCKRATIO, MAP = TD_MAP2 ( id INTEGER, vector_index SYSUDTLIB.Vector, clusterID BIGINT, Listing_Validity PERIOD(DATE) AS VALIDTIME) PRIMARY INDEX ( id );"


```sql
CREATE MULTISET TABLE DF120645.vectorstore_commercial_real_estate_temporal_centroids ,FALLBACK ,
     NO BEFORE JOURNAL,
     NO AFTER JOURNAL,
     CHECKSUM = DEFAULT,
     DEFAULT MERGEBLOCKRATIO,
     MAP = TD_MAP2
     (
      id INTEGER,
      vector_index SYSUDTLIB.Vector,
      clusterID BIGINT,
      Listing_Validity PERIOD(DATE) AS VALIDTIME)
PRIMARY INDEX ( id );
```

In [321]:
NONSEQUENCED VALIDTIME
INSERT INTO vectorstore_commercial_real_estate_temporal_centroids 
    (id, vector_index, clusterID, Listing_Validity)
SELECT 
    id, vector_index, clusterID,
    PERIOD(DATE '2025-06-01', DATE '9999-01-01')
FROM 
    vectorstore_commercial_real_estate_temporal_centroids_temp;

Success: 1541 rows affected

In [322]:
 set session validtime ( PERIOD(TIMESTAMP '2025-01-01 00:00:00.000', TIMESTAMP '2025-05-01 00:00:00.000'));

Success: 1 rows affected

**Ask** question based on temporal

In [257]:
WITH ClusterIDs AS (
    -- Fetch clusterID values dynamically
    SELECT reference_id FROM TD_VECTORDISTANCE(
                ON question_vector AS TargetTable DIMENSION
                ON (    
                        CURRENT VALIDTIME
                        SELECT td_clusterid_kmeans as clusterID, vector_index AS centroid 
                        FROM vectorstore_commercial_real_estate_temporal_model 
                        WHERE td_clusterid_kmeans IS NOT NULL) AS ReferenceTable
                USING
                TargetIDColumn('ID')
                TargetFeatureColumns('Embedding')
                RefIDColumn('clusterID')
                RefFeatureColumns('centroid')
                DistanceMeasure('EUCLIDEAN')
                OutputSimilarity('t')
                Topk(3)
                LargeReferenceInput('t')
                EmbeddingSize(1536)
                ) AS dt 
)
CURRENT VALIDTIME
SELECT t.id, t."title", t.price, vd.similarity
FROM df120645.commercial_real_estate_temporal t
JOIN (
    SELECT reference_id, similarity
    FROM TD_VECTORDISTANCE(
        ON question_vector AS TargetTable DIMENSION
        ON (
            CURRENT VALIDTIME
            SELECT * 
            FROM vectorstore_commercial_real_estate_temporal_centroids 
            WHERE clusterID IN (SELECT reference_id FROM ClusterIDs)
        ) AS ReferenceTable
        USING
            TargetIDColumn('id')
            TargetFeatureColumns('embedding')
            RefIDColumn('id')
            RefFeatureColumns('vector_index')
            DistanceMeasure('euclidean')
            OutputSimilarity('t')
            Topk(10)
            EmbeddingSize(1536)
            LargeReferenceInput('t')
    ) AS dt1
) vd
ON t.id = vd.reference_id
ORDER BY vd.similarity DESC;

,id,title,price,similarity
1,68,"Suite 220-221, 111 Harrington Street, Sydney, nsw 2000 - For Sale - Offices - ID: 381123","$1,500,000",0.06693688177180236
2,68,"Suite 220-221, 111 Harrington Street, Sydney, nsw 2000 - For Sale - Offices - ID: 381123","$1,500,000",0.06693688177180236
3,68,"Suite 220-221, 111 Harrington Street, Sydney, nsw 2000 - For Sale - Offices - ID: 381123","$1,500,000",0.06693688177180236
4,757,"57C Carrington Street, Palmyra, WA 6157 - For Sale - Retail",Contact Agent,0.06066465198962686
5,757,"57C Carrington Street, Palmyra, WA 6157 - For Sale - Retail",Contact Agent,0.06066465198962686
6,757,"57C Carrington Street, Palmyra, WA 6157 - For Sale - Retail",Contact Agent,0.06066465198962686
7,899,"7 Durham Street, Mount Druitt, NSW 2770 - For Sale - Other",Contact the exclusive agents,0.06058482696977657
8,899,"7 Durham Street, Mount Druitt, NSW 2770 - For Sale - Other",Contact the exclusive agents,0.06058482696977657
9,899,"7 Durham Street, Mount Druitt, NSW 2770 - For Sale - Other",Contact the exclusive agents,0.06058482696977657
10,382,"Dunolly, VIC 3472 - For Sale - Retail","$309,000",0.06029512969880231


# Fini